# שיעור 13 - זיכרון סוכן


## הגדרה

מחברת זו מדגימה כיצד לבנות סוכן הזמנות נסיעות עם **זיכרון מתמשך** באמצעות **מסגרת הסוכן של מיקרוסופט** (MAF).

תלמד כיצד סוגי זיכרון סוכן שונים — עבודה, קצר טווח, וארוך טווח — משפיעים על האופן שבו סוכן שומר ומשתמש במידע במהלך שיחות.

**דרישות מקדימות:**
- פרויקט Azure AI Foundry עם מודל צ’אט פרוס (למשל `gpt-4o-mini`).
- מחובר באמצעות Azure CLI — הפעל `az login` בטרמינל שלך.
- `AZURE_AI_PROJECT_ENDPOINT` — נקודת הקצה של פרויקט Azure AI Foundry שלך.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — שם המודל שהופעל.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## סוגי זיכרון סוכן

סוכני בינה מלאכותית יכולים להשתמש בסוגים שונים של זיכרון, כאשר כל סוג משרת מטרה שונה:

### זיכרון עבודה
שרשור השיחה עצמו — ההודעות המוחלפות במפגש יחיד. הסוכן יכול להתייחס להודעות קודמות באותו השרשור כדי לשמור על עקביות. ב-MAF זה נוצר באמצעות **`agent.create_session()`**, שמחזיר `AgentSession`.

### זיכרון לטווח קצר
מידע שנשמר לאורך ביצוע משימה או מפגש אך אינו מתועד באופן קבוע. לדוגמה, הסוכן עשוי לצבור עובדות במהלך שיחה מתוכננת מרובת שלבים ולהשתמש בהן ליצירת מסלול סופי.

### זיכרון לטווח ארוך
העדפות ועובדות שנשמרות **בין מפגשים**. משתמש חוזר לא צריך לחזור ולציין את הגבלות התזונה או סגנון הנסיעה שלו. בדרך כלל זיכרון לטווח ארוך מגובה ע"י מאגר חיצוני — מסד נתונים, קובץ או אינדקס וקטורי — ומוצג לסוכן באמצעות כלים.


## זיכרון עבודה עם סשנים

הצורה הפשוטה ביותר של זיכרון היא סשן שיחה. כאשר אתה מעביר את אותו הסשן (נוצר דרך `agent.create_session()`) לקריאות `agent.run()` עוקבות, הסוכן רואה את ההיסטוריה המלאה של שיחה זו ויכול להזכיר פרטים מוקדמים.

בוא ניצור סוכן נסיעות ונציג זיכרון עבודה.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

הסוכן זכר נכונה את התקציב כיוון ששתי ההודעות משותפות לאותה סשן. זו **זיכרון עבודה** — היא קיימת רק לאורך החיים של הסשן.

### מה קורה עם שיחה חדשה?

אם ניצור סשן **חדש**, לסוכן אין זיכרון של השיחה הקודמת:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## תבנית זיכרון לטווח ארוך

כדי לזכור העדפות משתמש **במהלך סשנים**, אנחנו צריכים מאגר קבוע שחי מחוץ לשרשור השיחה. הסוכן ניגש למאגר זה דרך **כלים** — פונקציות שהוא יכול לקרוא כדי לשמור ולשלוף מידע.

להלן מימוש פשוט של מאגר העדפות בזיכרון פנימי (בייצור תתמוך בבסיס נתונים או אינדקס וקטורי) וחשיפתו ככלים שהסוכן יכול להשתמש בהם.

### ארכיטקטורה
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### תרחיש 1 — משתמש בפעם הראשונה מזמין טיול ליום השנה

שרה מבקרת בפעם הראשונה. הסוכן צריך לשמור את ההעדפות שלה באמצעות הכלים ולהשתמש בהם כדי להמליץ על מלונות.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### תרחיש 2 — שרה חוזרת לאחר מספר שבועות

שרה מתחילה **שרשור חדש לחלוטין** (מדמה מושב חדש). זיכרון העבודה ריק, אך מאגר ההעדפות לטווח הארוך עדיין שומר את המידע שלה. הסוכן צריך לשלוף אותו ולהשתמש בו להתאמה אישית של ההמלצות.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## סיכום

במדריך זה למדתם שלושה סוגי זיכרון סוכן ואיך לממש אותם עם מסגרת הסוכן של מיקרוסופט:

| סוג הזיכרון | מנגנון MAF | משך חיים |
|---|---|---|
| **עובד** | `agent.create_session()` | שיחה יחידה |
| **קצר טווח** | הקשר מצטבר בתוך שרשור | משימה / סשן יחיד |
| **ארוך טווח** | מאגר חיצוני שניגשים אליו באמצעות פונקציות `@tool` | מעבר לסשנים |

### נקודות מפתח
1. **`agent.create_session()`** מספק זיכרון עובד — הסוכן רואה את כל היסטוריית השיחה במסגרת הסשן.
2. **סשנים חדשים מפסידים הקשר** — ללא זיכרון ארוך טווח הסוכן לא יכול לזכור שיחות קודמות.
3. **פונקציות `@tool`** גשר פער — הן מאפשרות לסוכן לשמור ולשלוף מידע מתוך מאגר מתמיד.
4. **התאמה אישית משתפרת עם הזמן** — ככל שנאגרות העדפות, ההמלצות של הסוכן משתפרות.

### יישומים בעולם האמיתי
- **שירות לקוחות**: זיכרון היסטוריה והעדפות לקוח
- **עוזרים אישיים**: שמירת הקשר על פני ימים או שבועות
- **בריאות**: מעקב אחר מידע והעדפות מטופל
- **מסחר אלקטרוני**: קניות מותאמות אישית על בסיס היסטוריה

### השלבים הבאים
- החלפת מילון בזיכרון במאגר נתונים או מאגר וקטורים (לדוגמה Azure AI Search)
- הוספת תוקף לזיכרון למידע רגיש לזמן
- בניית מערכות רב-סוכן עם זיכרון משותף
- שימוש במחברת Cognee לזיכרון מגובה בגרף ידע


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**הצהרת אחריות**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים לכלול שגיאות או אי-דיוקים. המסמך המקורי בשפה המקורית שלו יש להיחשב כמקור הסמכותי. למידע קריטי מומלץ להשתמש בתרגום מקצועי שנעשה על ידי אדם. איננו אחראים לכל אי-הבנה או פרשנות שגויה הנובעת משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
